# 🚀 Zara Visual Product Recognition - The "Beast" Pipeline
**Hardware:** 2x Tesla T4 GPUs | **Estrategia:** YOLOv8 Seg + SigLIP 384px + Contrastive Fine-Tuning + Heurísticas Avanzadas

## 🧱 Bloque 1: Setup, Data Hacking & Hard Constraints
En esta fase inicial:
1. Instalamos las dependencias pesadas que usaremos más adelante (`ultralytics`, `timm`, `open_clip_torch`).
2. Extraemos inteligentemente los **SKUs** y **Timestamps** de las URLs de las imágenes.
3. Aplicamos el **Noise Filter**: Eliminamos 53 categorías de productos que son estadísticamente invisibles (perfumes, velas, etc.).
4. Extraemos las **Hard Constraints (Lógica de Secciones)**: Analizamos el `train_df` para aprender qué categorías están permitidas estrictamente en cada sección comercial, bloqueando cruces absurdos (ej. un vestido en la sección de hombre).

In [1]:
# ==========================================
# BLOQUE 1: SETUP, DATA HACKING & HARD CONSTRAINTS
# ==========================================
# 1. Instalación de artillería pesada (silenciada para no ensuciar el notebook)
!pip install -q ultralytics timm open_clip_torch faiss-gpu

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings

# Silenciar warnings
warnings.filterwarnings('ignore')

print("=== 1. INICIANDO DATA ENGINEERING ===")

# 2. Configuración de Rutas en Kaggle
DATA_DIR = Path('/kaggle/input/datasets/nicolsaller/inditex')
WORK_DIR = Path('/kaggle/working')
IMG_DIR = WORK_DIR / 'images'
IMG_DIR.mkdir(parents=True, exist_ok=True)

# 3. Carga de los Datasets
try:
    bundles_df = pd.read_csv(DATA_DIR / 'bundles_dataset.csv')
    products_df = pd.read_csv(DATA_DIR / 'product_dataset.csv')
    train_df = pd.read_csv(DATA_DIR / 'bundles_product_match_train.csv')
    test_df = pd.read_csv(DATA_DIR / 'bundles_product_match_test.csv')
    print("✅ CSVs cargados correctamente.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encuentran los datos en {DATA_DIR}. ¡Revisa la ruta en Kaggle!")

# 4. Hacking de Metadatos (Extracción Vectorizada)
print("Extrayendo SKUs y Timestamps...")
sku_pattern = r'/([A-Z0-9]{8,15}(?:-\d+)?)-[pe]'
bundles_df['sku'] = bundles_df['bundle_image_url'].str.extract(sku_pattern, expand=False)
products_df['sku'] = products_df['product_image_url'].str.extract(sku_pattern, expand=False)

ts_pattern = r'ts=(\d+)'
bundles_df['ts'] = bundles_df['bundle_image_url'].str.extract(ts_pattern, expand=False).astype(float)
products_df['ts'] = products_df['product_image_url'].str.extract(ts_pattern, expand=False).astype(float)

# 5. Filtro de Ruido (Noise Filter)
NOISE_CATEGORIES = {
    'BABY ACCESORIES', 'BABY BODY', 'BABY OUTFIT', 'BABY OVERALL', 'BABY PANTY/UNDERP.', 
    'BABY POLO SHIRT', 'BABY PYJAMA', 'BABY ROMPER SUIT', 'BABY SWIMSUIT', 'BATHROBE/DRES.GOWN', 
    'BEACH SANDAL', 'BODY OIL', 'BOOKS', 'BOW TIE/CUMMERBAND', 'CANDLE', 'EAU DE COLOGNE', 
    'EAU DE TOILETTE', 'ENSEMBLE..SET', 'HAIR COSMETICS', 'HAND CREAM', 'HOME SHOES', 'LIP BALM', 
    'LIP SUNSCREEN', 'MATCHES', 'MOISTURISING CREAM', 'NAIL COSMETICS', 'NAIL POLISH', 'NEWBORN', 
    'NEWBORN TRICOT', 'PARKA', 'PERFUME', 'PERFUMED SOAP', 'POWDER BRUSH-PUFF', 'SHAMPOO', 
    'STATIONERY', 'SUSPENDERS', 'TOWEL', 'UMBRELLA', 'UNIFORM', 'BABY LEGGINGS', 'BABY SOCKS', 
    'BABY TRACKSUIT', 'BABY WAISTCOAT', 'BABY WIND-JACKET', 'BIB OVERALL', 'EAU DE PARFUM', 
    'EYES CONTOUR', 'PURSE WALLET', 'SLEEVELESS PAD. JACKET', 'SPORTY SANDAL', 'STOCKINGS-TIGHTS', 
    'UNDERWEAR', 'WALLETS'
}

initial_count = len(products_df)
products_df = products_df[~products_df['product_description'].isin(NOISE_CATEGORIES)].copy()
print(f"🗑️ Ruido eliminado: {initial_count - len(products_df)} productos invisibles descartados.")
print(f"📦 Catálogo final útil: {len(products_df)} productos.")

# 6. Mapeos ultra-rápidos en memoria
bundle_section_map = dict(zip(bundles_df['bundle_asset_id'], bundles_df['bundle_id_section']))
product_desc_map = dict(zip(products_df['product_asset_id'], products_df['product_description']))

# 7. Extracción de Lógica de Negocio (Hard Constraints)
print("\nAprendiendo reglas de cruce de secciones...")
category_allowed_sections = defaultdict(set)

for _, row in train_df.iterrows():
    b_id = row['bundle_asset_id']
    p_id = row['product_asset_id']
    
    sec = bundle_section_map.get(b_id)
    desc = product_desc_map.get(p_id)
    
    if sec and desc and pd.notna(sec) and pd.notna(desc):
        category_allowed_sections[desc].add(sec)

print(f"Reglas aprendidas para {len(category_allowed_sections)} categorías distintas.")

# Función maestra que usaremos al final para aniquilar falsos positivos
def is_valid_for_section(product_id, bundle_id):
    sec = bundle_section_map.get(bundle_id)
    desc = product_desc_map.get(product_id)
    
    if not sec or not desc: return True # Si nos falta info, somos conservadores y lo permitimos
    if desc not in category_allowed_sections: return True # Cold start
    
    return sec in category_allowed_sections[desc]

print("\nBloque 1 Finalizado con éxito. Entorno preparado.")

ERROR: Ignored the following versions that require a different python version: 8.0.10 Requires-Python >=3.7,<=3.11; 8.0.11 Requires-Python >=3.7,<=3.11; 8.0.12 Requires-Python >=3.7,<=3.11; 8.0.13 Requires-Python >=3.7,<=3.11; 8.0.14 Requires-Python >=3.7,<=3.11; 8.0.15 Requires-Python >=3.7,<=3.11; 8.0.16 Requires-Python >=3.7,<=3.11; 8.0.17 Requires-Python >=3.7,<=3.11; 8.0.18 Requires-Python >=3.7,<=3.11; 8.0.19 Requires-Python >=3.7,<=3.11; 8.0.20 Requires-Python >=3.7,<=3.11; 8.0.21 Requires-Python >=3.7,<=3.11; 8.0.22 Requires-Python >=3.7,<=3.11; 8.0.23 Requires-Python >=3.7,<=3.11; 8.0.24 Requires-Python >=3.7,<=3.11; 8.0.25 Requires-Python >=3.7,<=3.11; 8.0.26 Requires-Python >=3.7,<=3.11; 8.0.27 Requires-Python >=3.7,<=3.11; 8.0.28 Requires-Python >=3.7,<=3.11; 8.0.29 Requires-Python >=3.7,<=3.11; 8.0.30 Requires-Python >=3.7,<=3.11; 8.0.31 Requires-Python >=3.7,<=3.11; 8.0.32 Requires-Python >=3.7,<=3.11; 8.0.33 Requires-Python >=3.7,<=3.11; 8.0.34 Requires-Python >=3.7,<=3.

In [8]:
# ==========================================
# BLOQUE 1.5: DESCARGA ASÍNCRONA DE IMÁGENES
# ==========================================
import asyncio
import aiohttp
import nest_asyncio
from tqdm.asyncio import tqdm
from pathlib import Path

nest_asyncio.apply()

print("=== 1.5 DESCARGANDO IMÁGENES AL NUEVO ENTORNO ===")

# Crear directorios
BUNDLES_IMG_DIR = WORK_DIR / 'images/bundles'
PRODUCTS_IMG_DIR = WORK_DIR / 'images/products'
BUNDLES_IMG_DIR.mkdir(parents=True, exist_ok=True)
PRODUCTS_IMG_DIR.mkdir(parents=True, exist_ok=True)

async def download_image(session, url, path, semaphore):
    if path.exists() and path.stat().st_size > 0:
        return True
    try:
        async with semaphore:
            async with session.get(url, timeout=15) as response:
                if response.status == 200:
                    content = await response.read()
                    with open(path, 'wb') as f:
                        f.write(content)
                    return True
                return False
    except:
        return False

async def main_download():
    tasks = []
    semaphore = asyncio.Semaphore(100) # 100 conexiones simultáneas
    
    async with aiohttp.ClientSession() as session:
        # Bundles
        for _, row in bundles_df.iterrows():
            img_path = BUNDLES_IMG_DIR / f"{row['bundle_asset_id']}.jpg"
            tasks.append(download_image(session, row['bundle_image_url'], img_path, semaphore))
            
        # Productos (solo los del catálogo filtrado)
        for _, row in products_df.iterrows():
            img_path = PRODUCTS_IMG_DIR / f"{row['product_asset_id']}.jpg"
            tasks.append(download_image(session, row['product_image_url'], img_path, semaphore))
            
        print(f"Lanzando {len(tasks)} descargas...")
        results = await tqdm.gather(*tasks, desc="Descargando")
        print(f"✅ Exitosos: {sum(r for r in results if r)} | Fallos: {len(tasks) - sum(r for r in results if r)}")

# Ejecutar descarga
asyncio.run(main_download())

=== 1.5 DESCARGANDO IMÁGENES AL NUEVO ENTORNO ===
Lanzando 28370 descargas...


Descargando: 100%|██████████| 28370/28370 [00:15<00:00, 1872.68it/s] 


✅ Exitosos: 28370 | Fallos: 0


## ✂️ Bloque 2: YOLOv8 Segmentation & Spatial Mapping

Para solucionar la "ceguera" del modelo y el ruido del fondo (suelo, paredes), usamos **YOLOv8-Seg**:
1. Detecta la silueta exacta de la persona en el *bundle*.
2. Aplica una máscara que convierte todo el fondo de la imagen en negro puro (RGB 0,0,0).
3. Divide anatómicamente el bounding box de la persona en 4 zonas: `HEAD`, `TORSO`, `LEGS`, `FEET`.
4. Guarda estos recortes limpios en disco junto con la vista `GLOBAL`, etiquetando la zona de la que provienen para aplicar las **Hard Constraints espaciales** después.

In [4]:
!pip install -q ultralytics 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.8 MB/s eta 0:00:00a 0:00:01


In [9]:
# ==========================================
# BLOQUE 2: YOLOv8 SEGMENTATION & SPATIAL MAPPING
# ==========================================
from ultralytics import YOLO
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import os
import pandas as pd

print("=== 2. INICIANDO SEGMENTACIÓN CON YOLOv8 ===")

# 1. Crear directorio para guardar los recortes de forma segura y no saturar la RAM
CROPS_DIR = WORK_DIR / 'smart_crops'
CROPS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Cargar el modelo YOLOv8 de segmentación (Usamos 'm' para un balance perfecto entre velocidad y precisión)
# Se descargará automáticamente de los servidores de Ultralytics
yolo_model = YOLO("yolov8m-seg.pt") 

# 3. Preparar lista de bundles a procesar (Train + Test)
# Solo procesaremos los bundles únicos para no repetir trabajo
bundles_to_process = list(set(test_df['bundle_asset_id'].tolist() + train_df['bundle_asset_id'].tolist()))
print(f"Total de bundles a procesar: {len(bundles_to_process)}")

crops_metadata = []

def process_and_save_bundle(bundle_id):
    img_path = str(BUNDLES_IMG_DIR / f"{bundle_id}.jpg") if 'BUNDLES_IMG_DIR' in globals() else str(WORK_DIR / f"images/bundles/{bundle_id}.jpg")
    
    if not os.path.exists(img_path):
        return
    
    img = cv2.imread(img_path)
    if img is None: return
    
    h_img, w_img = img.shape[:2]
    
    # Inferencia (classes=[0] para detectar solo personas)
    results = yolo_model(img, classes=[0], conf=0.3, verbose=False)
    
    # --- 1. Guardar la imagen Global ---
    global_path = CROPS_DIR / f"{bundle_id}_GLOBAL.jpg"
    cv2.imwrite(str(global_path), img)
    crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'GLOBAL', 'crop_path': str(global_path)})
    
    # Procesar detecciones
    for r in results:
        # Si encuentra máscara y bounding box
        if r.masks is not None and r.boxes is not None:
            # Coger la primera persona detectada (la más segura)
            mask = r.masks.data[0].cpu().numpy()
            mask = cv2.resize(mask, (w_img, h_img))
            
            # Remover fondo (lo ponemos a negro)
            img_bg_removed = img.copy()
            img_bg_removed[mask < 0.5] = 0
            
            # Bounding box de la persona
            x1, y1, x2, y2 = map(int, r.boxes.xyxy[0])
            person_crop = img_bg_removed[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)]
            
            if person_crop.size == 0: continue
            
            h_p, w_p = person_crop.shape[:2]
            
            # --- 2. Recortes Anatómicos (Guardamos en disco) ---
            
            # A) HEAD (0% - 20%)
            head = person_crop[0:int(h_p*0.2), 0:w_p]
            head_path = CROPS_DIR / f"{bundle_id}_HEAD.jpg"
            cv2.imwrite(str(head_path), head)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'HEAD', 'crop_path': str(head_path)})
            
            # B) TORSO (20% - 60%)
            torso = person_crop[int(h_p*0.2):int(h_p*0.6), 0:w_p]
            torso_path = CROPS_DIR / f"{bundle_id}_TORSO.jpg"
            cv2.imwrite(str(torso_path), torso)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'TORSO', 'crop_path': str(torso_path)})
            
            # C) LEGS (50% - 85%) - Solapamos un poco con el torso por si las camisetas son largas
            legs = person_crop[int(h_p*0.5):int(h_p*0.85), 0:w_p]
            legs_path = CROPS_DIR / f"{bundle_id}_LEGS.jpg"
            cv2.imwrite(str(legs_path), legs)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'LEGS', 'crop_path': str(legs_path)})
            
            # D) FEET (80% - 100%) - Especial para zapatos, sin suelo ruidoso
            feet = person_crop[int(h_p*0.8):h_p, 0:w_p]
            feet_path = CROPS_DIR / f"{bundle_id}_FEET.jpg"
            cv2.imwrite(str(feet_path), feet)
            crops_metadata.append({'bundle_asset_id': bundle_id, 'zone': 'FEET', 'crop_path': str(feet_path)})
            
            # Solo procesamos la persona principal para evitar recortes de gente de fondo
            break 

# 4. Ejecutar el procesamiento
for b_id in tqdm(bundles_to_process, desc="Segmentando y Recortando"):
    process_and_save_bundle(b_id)

# 5. Convertir a DataFrame para fácil acceso después
crops_df = pd.DataFrame(crops_metadata)
print(f"\n✅ Segmentación completada. {len(crops_df)} recortes generados y guardados en disco.")

=== 2. INICIANDO SEGMENTACIÓN CON YOLOv8 ===
Total de bundles a procesar: 2331


Segmentando y Recortando:   0%|          | 0/2331 [00:00<?, ?it/s]


✅ Segmentación completada. 11655 recortes generados y guardados en disco.


## 🦅 Bloque 3: SigLIP-384 High-Res Embeddings

Reemplazamos la visión estándar por **SigLIP-SO400M a 384px**. 
1. Cargamos el modelo en DataParallel usando las 2 GPUs T4 para destruir los tiempos de inferencia.
2. Pasamos los ~26.000 productos del catálogo.
3. Pasamos los 11.655 recortes anatómicos de los *bundles* (GLOBAL, HEAD, TORSO, LEGS, FEET).
4. Guardamos todo en diccionarios anidados para que la búsqueda posterior sea instantánea.

In [14]:
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


In [15]:
# ==========================================
# BLOQUE 3: HIGH-RES EMBEDDINGS (SigLIP 384px)
# ==========================================
import torch
import open_clip
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import os

print("=== 3. EXTRACCIÓN DE EMBEDDINGS ALTA RESOLUCIÓN ===")

# 1. Cargar SigLIP 384px (El "Ojo de Águila")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cargando modelo pesado SigLIP SO400M (Esto tardará 1 minuto)...")

# open_clip nos da el modelo y el preprocesador que redimensiona a 384x384
model, _, preprocess = open_clip.create_model_and_transforms('ViT-SO400M-14-SigLIP-384', pretrained='webli')

if torch.cuda.device_count() > 1:
    print(f"🔥 ¡Activando DataParallel en {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
model = model.to(device)
model.eval()

# 2. Clase Dataset Universal
class HighResImageDataset(Dataset):
    def __init__(self, image_paths, ids, preprocess):
        self.image_paths = image_paths
        self.ids = ids
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        item_id = self.ids[idx]
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = self.preprocess(img)
            return img_tensor, item_id
        except Exception:
            # Fallback en caso de imagen corrupta
            return torch.zeros((3, 384, 384)), item_id

# 3. Procesar Productos
valid_products = products_df[products_df['product_asset_id'].apply(lambda x: os.path.exists(PRODUCTS_IMG_DIR / f"{x}.jpg"))]
p_paths = [PRODUCTS_IMG_DIR / f"{pid}.jpg" for pid in valid_products['product_asset_id']]
p_ids = valid_products['product_asset_id'].tolist()

prod_dataset = HighResImageDataset(p_paths, p_ids, preprocess)
# Usamos un batch size un poco más bajo (64) porque la resolución 384 ocupa mucha VRAM
prod_loader = DataLoader(prod_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

product_embeddings = {}
with torch.no_grad():
    for images, ids in tqdm(prod_loader, desc="Inferencia Productos (SigLIP)"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            # El método encode_image es el estándar en open_clip
            features = model.module.encode_image(images) if isinstance(model, torch.nn.DataParallel) else model.encode_image(images)
            
        # Normalización L2 imprescindible para el producto punto posterior
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().float().numpy()
        
        for i, p_id in enumerate(ids):
            product_embeddings[p_id] = features[i]

# 4. Procesar Recortes de Bundles
crop_files = list(CROPS_DIR.glob("*.jpg"))
c_paths = [str(p) for p in crop_files]
# c_ids tendrá formato "B_1234_HEAD", "B_1234_GLOBAL", etc.
c_ids = [p.stem for p in crop_files] 

crop_dataset = HighResImageDataset(c_paths, c_ids, preprocess)
crop_loader = DataLoader(crop_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

bundle_crop_embeddings = {}
with torch.no_grad():
    for images, ids in tqdm(crop_loader, desc="Inferencia Crops Anatómicos"):
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            features = model.module.encode_image(images) if isinstance(model, torch.nn.DataParallel) else model.encode_image(images)
            
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().float().numpy()
        
        for i, crop_id in enumerate(ids):
            # Parseamos el ID (ej: "B_c848ff0bf533_FEET" -> b_id="B_c848ff0bf533", zone="FEET")
            parts = crop_id.rsplit('_', 1)
            if len(parts) == 2:
                b_id, zone = parts
                if b_id not in bundle_crop_embeddings:
                    bundle_crop_embeddings[b_id] = {}
                bundle_crop_embeddings[b_id][zone] = features[i]

print(f"\nExtracción completada.")
print(f"Vectores de Productos: {len(product_embeddings)}")
print(f"Bundles con Multi-Vectores: {len(bundle_crop_embeddings)}")

=== 3. EXTRACCIÓN DE EMBEDDINGS ALTA RESOLUCIÓN ===
Cargando modelo pesado SigLIP SO400M (Esto tardará 1 minuto)...


open_clip_model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

🔥 ¡Activando DataParallel en 2 GPUs!


Inferencia Productos (SigLIP):   0%|          | 0/407 [00:00<?, ?it/s]

Inferencia Crops Anatómicos:   0%|          | 0/183 [00:00<?, ?it/s]


Extracción completada.
Vectores de Productos: 26039
Bundles con Multi-Vectores: 2331


## 🧠 Bloque 4: Contrastive Fine-Tuning (K-Fold)

SigLIP extrae vectores de **1152 dimensiones** de altísima calidad, pero su espacio latente es genérico. 
Entrenamos una **Projection Head** (Red Neuronal Residual) usando Contrastive Learning (InfoNCE Loss) para adaptar estos vectores a la lógica de *outfits* de Zara.
- **Entrenamiento:** Usamos la vista `GLOBAL` del bundle frente a la foto del producto.
- **Validación:** 5-Fold Cross Validation para evitar el sobreajuste y quedarnos con los pesos matemáticamente perfectos.
- **Reproyección:** Pasamos *todos* los recortes de YOLO y los productos por esta red para crear el espacio vectorial definitivo.

In [20]:
# ==========================================
# BLOQUE 4.1: CROP-AWARE FINE-TUNING + EARLY STOPPING
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from sklearn.model_selection import KFold
import copy

print("=== 4.1 INICIANDO CROP-AWARE FINE-TUNING ===")

# 1. Mapeo de Categoría a Zona de YOLO
def get_ideal_zone(desc):
    if not isinstance(desc, str): return 'GLOBAL'
    desc = desc.upper()
    
    if desc in {'HAT/HEADBAND', 'GLASSES', 'SCARF', 'EARRINGS', 'NECKLACE'}: return 'HEAD'
    if desc in {'SHOES', 'MOCCASINS', 'SANDAL', 'SNEAKERS', 'FLAT SHOES', 'HEELED SHOES', 'ANKLE BOOT', 'HEELED BOOT', 'TRAINERS', 'RAIN BOOT', 'SOCKS'}: return 'FEET'
    if desc in {'TROUSERS', 'JEANS', 'SKIRT', 'BERMUDAS', 'SHORTS', 'LEGGINGS', 'CARGO PANTS'}: return 'LEGS'
    if desc in {'SHIRT', 'T-SHIRT', 'SWEATER', 'SWEATSHIRT', 'JACKET', 'COAT', 'PUFFER', 'WAISTCOAT', 'TOP', 'BODY', 'BLOUSE'}: return 'TORSO'
    
    return 'GLOBAL'

# 2. Arquitectura con Temperatura dinámica (Sota Trick)
class ProjectionHead(nn.Module):
    def __init__(self, embedding_dim=1152, hidden_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.2), # Aumentamos dropout para combatir el overfitting
            nn.Linear(hidden_dim, embedding_dim)
        )
        # Parámetro aprendible para la temperatura de InfoNCE
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        
    def forward(self, x):
        projected = x + self.net(x)
        return F.normalize(projected, p=2, dim=-1)

# 3. Dataset Inteligente (Sabe qué recorte mirar)
class CropAwareDataset(Dataset):
    def __init__(self, pairs, bundle_crop_dict, product_emb_dict, product_desc_map):
        self.pairs = pairs
        self.b_dict = bundle_crop_dict
        self.p_dict = product_emb_dict
        self.desc_map = product_desc_map

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        b_id, p_id = self.pairs[idx]
        b_crops = self.b_dict[b_id]
        
        # Lógica Anatómica: Buscar el crop perfecto
        desc = self.desc_map.get(p_id, '')
        ideal_zone = get_ideal_zone(desc)
        
        # Si el crop ideal existe lo usamos, si no, fallback a GLOBAL, y si no, al primero que haya
        if ideal_zone in b_crops: b_emb = b_crops[ideal_zone]
        elif 'GLOBAL' in b_crops: b_emb = b_crops['GLOBAL']
        else: b_emb = list(b_crops.values())[0]
            
        p_emb = self.p_dict[p_id]
        return torch.tensor(b_emb, dtype=torch.float32), torch.tensor(p_emb, dtype=torch.float32)

# Pares válidos
valid_pairs = np.array([(row['bundle_asset_id'], row['product_asset_id']) 
                        for _, row in train_df.iterrows() 
                        if row['bundle_asset_id'] in bundle_crop_embeddings 
                        and row['product_asset_id'] in product_embeddings])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def info_nce_loss(model, bundle_feats, product_feats):
    # Usamos la temperatura optimizada por la red
    logit_scale = model.logit_scale.exp()
    logits = logit_scale * torch.matmul(bundle_feats, product_feats.T)
    labels = torch.arange(len(bundle_feats)).to(device)
    return F.cross_entropy(logits, labels)

best_val_loss_global = float('inf')
best_model_state = None
best_fold = -1
EPOCHS = 20

for fold, (train_idx, val_idx) in enumerate(kf.split(valid_pairs)):
    print(f"\n--- FOLD {fold + 1}/5 ---")
    
    train_dataset = CropAwareDataset(valid_pairs[train_idx], bundle_crop_embeddings, product_embeddings, product_desc_map)
    val_dataset = CropAwareDataset(valid_pairs[val_idx], bundle_crop_embeddings, product_embeddings, product_desc_map)
    
    train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False, drop_last=True)
    
    model = ProjectionHead().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
    
    best_val_loss_fold = float('inf')
    best_epoch_weights = None
    patience = 3 # Early stopping
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for b_emb, p_emb in train_loader:
            b_emb, p_emb = b_emb.to(device), p_emb.to(device)
            optimizer.zero_grad()
            loss = info_nce_loss(model, model(b_emb), model(p_emb))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for b_emb, p_emb in val_loader:
                b_emb, p_emb = b_emb.to(device), p_emb.to(device)
                loss = info_nce_loss(model, model(b_emb), model(p_emb))
                val_loss += loss.item()
                
        avg_train = train_loss / len(train_loader) if len(train_loader) > 0 else 0
        avg_val = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        
        print(f"  Epoch {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")
            
        if avg_val < best_val_loss_fold and avg_val > 0:
            best_val_loss_fold = avg_val
            best_epoch_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        # Si empeora 3 veces seguidas, cortamos
        if patience_counter >= patience:
            print(f"  🛑 Early Stopping en la época {epoch+1} para evitar Overfitting.")
            break
            
    # Chequeo global del mejor fold
    if best_val_loss_fold < best_val_loss_global:
        best_val_loss_global = best_val_loss_fold
        best_model_state = copy.deepcopy(best_epoch_weights)
        best_fold = fold + 1

print(f"\n🏆 Mejor modelo: Fold {best_fold} con Val Loss: {best_val_loss_global:.4f}")

# 4. Reproyectar
print("\n--- RE-PROYECTANDO EMBEDDINGS ---")
final_model = ProjectionHead().to(device)
final_model.load_state_dict(best_model_state)
final_model.eval()

projected_bundle_crops = {}
projected_products = {}

with torch.no_grad():
    for b_id, zones_dict in bundle_crop_embeddings.items():
        projected_bundle_crops[b_id] = {}
        for zone, emb in zones_dict.items():
            t_emb = torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(device)
            projected_bundle_crops[b_id][zone] = final_model(t_emb).squeeze(0).cpu().numpy()
            
    p_ids = list(product_embeddings.keys())
    p_matrix = torch.tensor(np.vstack(list(product_embeddings.values())), dtype=torch.float32).to(device)
    
    batch_size = 2048
    new_p_matrix = []
    for i in range(0, len(p_matrix), batch_size):
        batch = p_matrix[i:i+batch_size]
        new_p_matrix.append(final_model(batch).cpu().numpy())
        
    new_p_matrix = np.vstack(new_p_matrix)
    for i, pid in enumerate(p_ids):
        projected_products[pid] = new_p_matrix[i]

print("✅ Fine-Tuning Anatómico completado. Vectores listos.")

=== 4.1 INICIANDO CROP-AWARE FINE-TUNING ===

--- FOLD 1/5 ---
  Epoch 01 | Train: 4.8692 | Val: 3.4949
  Epoch 02 | Train: 3.4420 | Val: 3.0823
  Epoch 03 | Train: 2.9527 | Val: 2.8167
  Epoch 04 | Train: 2.6348 | Val: 2.6777
  Epoch 05 | Train: 2.4221 | Val: 2.5873
  Epoch 06 | Train: 2.2444 | Val: 2.5242
  Epoch 07 | Train: 2.1030 | Val: 2.4793
  Epoch 08 | Train: 1.9834 | Val: 2.4496
  Epoch 09 | Train: 1.8791 | Val: 2.4179
  Epoch 10 | Train: 1.8028 | Val: 2.4033
  Epoch 11 | Train: 1.7301 | Val: 2.3985
  Epoch 12 | Train: 1.6595 | Val: 2.3795
  Epoch 13 | Train: 1.5877 | Val: 2.3725
  Epoch 14 | Train: 1.5358 | Val: 2.3739
  Epoch 15 | Train: 1.4761 | Val: 2.3726
  Epoch 16 | Train: 1.4447 | Val: 2.3655
  Epoch 17 | Train: 1.3967 | Val: 2.3763
  Epoch 18 | Train: 1.3476 | Val: 2.3743
  Epoch 19 | Train: 1.3094 | Val: 2.3780
  🛑 Early Stopping en la época 19 para evitar Overfitting.

--- FOLD 2/5 ---
  Epoch 01 | Train: 4.9764 | Val: 3.4966
  Epoch 02 | Train: 3.5077 | Val: 3.1232

## 🏆 Bloque 5: The Beast Search Engine & Final Submission

Fase final de ensamblaje. Cruzamos el espacio latente afinado con la lógica humana:
1. **Smart Matching:** Los candidatos se evalúan contra su recorte anatómico ideal.
2. **Hard Constraints:** Filtrado estricto por reglas de sección (Bloque 1).
3. **Metadata Boost:** Bonificaciones por Timestamp, SKU Prefix y Co-ocurrencia de catálogo.
4. **SKU Deduplication:** Forzamos la diversidad en el Top 15 eliminando variantes de color redundantes.

In [21]:
# ==========================================
# BLOQUE 5: THE BEAST SEARCH ENGINE & SUBMISSION
# ==========================================
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import base64
from IPython.display import HTML

print("=== 5. INICIANDO MOTOR DE BÚSQUEDA Y SCORING ===")

# 1. Diccionarios rápidos de metadatos
b_sku_map = bundles_df.set_index('bundle_asset_id')['sku'].to_dict()
b_ts_map = bundles_df.set_index('bundle_asset_id')['ts'].to_dict()
p_sku_map = products_df.set_index('product_asset_id')['sku'].to_dict()
p_ts_map = products_df.set_index('product_asset_id')['ts'].to_dict()

# 2. Grafo de Co-ocurrencia (Train)
cooccurrence = defaultdict(Counter)
train_bids = train_df.groupby('bundle_asset_id')['product_asset_id'].apply(list)
for pids in train_bids:
    for p1 in pids:
        for p2 in pids:
            if p1 != p2:
                cooccurrence[p1][p2] += 1

# 3. Preparar tensores de productos
all_p_ids = list(projected_products.keys())
all_p_matrix = np.vstack(list(projected_products.values())) # (26039, 1152)

test_bundles = test_df['bundle_asset_id'].unique()
final_submission = []

print(f"Evaluando {len(test_bundles)} bundles con lógica heurística...")

for b_id in test_bundles:
    if b_id not in projected_bundle_crops: continue
        
    b_crops = projected_bundle_crops[b_id]
    b_sku = str(b_sku_map.get(b_id, ''))
    b_ts = b_ts_map.get(b_id)
    
    candidates = []
    
    # Evaluar todos los productos usando operaciones vectorizadas masivas
    # Para hacerlo dinámico, calculamos la similitud contra el GLOBAL como base
    base_b_emb = b_crops.get('GLOBAL', list(b_crops.values())[0])
    base_similarities = np.dot(all_p_matrix, base_b_emb)
    
    # Pre-filtrar el top 1000 visual (para no hacer heurística pesada sobre 26k)
    top_1000_idx = np.argsort(base_similarities)[::-1][:1000]
    
    for idx in top_1000_idx:
        p_id = all_p_ids[idx]
        desc = product_desc_map.get(p_id, '')
        
        # --- REGLA 1: HARD CONSTRAINTS (Guillotina) ---
        if not is_valid_for_section(p_id, b_id):
            continue # Si no está permitido en esta sección, a la basura
            
        # --- REGLA 2: SMART MATCHING (Búsqueda Anatómica) ---
        ideal_zone = get_ideal_zone(desc)
        if ideal_zone in b_crops and ideal_zone != 'GLOBAL':
            # Recalculamos la similitud usando el recorte perfecto de YOLO
            visual_score = float(np.dot(projected_products[p_id], b_crops[ideal_zone]))
        else:
            visual_score = float(base_similarities[idx])
            
        # Escalar a 0-100 para aplicar los bonus
        score = visual_score * 100
        
        # --- REGLA 3: METADATA BOOST ---
        p_sku = str(p_sku_map.get(p_id, ''))
        p_ts = p_ts_map.get(p_id)
        is_accessory = desc in {'SHOES', 'SNEAKERS', 'HAT/HEADBAND', 'GLASSES', 'HAND BAG-RUCKSACK'}
        
        # Timestamp Boost
        if b_ts and p_ts and not pd.isna(b_ts) and not pd.isna(p_ts):
            diff_days = abs(b_ts - p_ts) / (86400 * 1000)
            if diff_days < 14: score += 15
            elif diff_days < 45: score += 5
            elif diff_days > 180 and not is_accessory: score -= 10
                
        # SKU Prefix Boost (Mismo modelo)
        if b_sku != 'nan' and p_sku != 'nan' and len(b_sku) >= 6 and len(p_sku) >= 6:
            if b_sku[:6] == p_sku[:6]:
                score += 30 # ¡Jackpot! Es una variante de la misma prenda
                
        candidates.append({'product_id': p_id, 'score': score, 'sku_base': p_sku[:8] if p_sku != 'nan' else p_id})

    # Ordenar por el score heurístico final
    candidates.sort(key=lambda x: x['score'], reverse=True)
    
    # --- REGLA 4: CO-OCURRENCE BOOST (2ª Pasada) ---
    # Cogemos los 2 mejores como "Anclas" seguras y subimos a sus amigos
    anchors = [c['product_id'] for c in candidates[:2]]
    for cand in candidates:
        for anchor in anchors:
            if cand['product_id'] in cooccurrence[anchor]:
                cand['score'] += cooccurrence[anchor][cand['product_id']] * 3
                
    # Re-ordenar tras la co-ocurrencia
    candidates.sort(key=lambda x: x['score'], reverse=True)
    
    # --- REGLA 5: DEDUPLICACIÓN DE SKU ---
    final_top_15 = []
    seen_skus = set()
    
    for cand in candidates:
        if cand['sku_base'] not in seen_skus:
            final_top_15.append(cand['product_id'])
            seen_skus.add(cand['sku_base'])
        if len(final_top_15) == 15:
            break
            
    # Añadir a la submission
    for p_id in final_top_15:
        final_submission.append({'bundle_asset_id': b_id, 'product_asset_id': p_id})

# 4. Generar y descargar
submission_df = pd.DataFrame(final_submission)
file_path = WORK_DIR / 'submission_beast_final.csv'
submission_df.to_csv(file_path, index=False)

print(f"\n✅ ¡SUBMISSION GENERADA! Total filas: {len(submission_df)}")

# Botón de Descarga
def create_download_link(file_path, title="📥 Descargar THE BEAST Submission", filename="submission_beast_final.csv"):
    with open(file_path, "rb") as f: data = f.read()
    b64 = base64.b64encode(data).decode()
    html = f'''<a href="data:text/csv;base64,{b64}" download="{filename}" 
               style="display: inline-block; padding: 15px 25px; background-color: #FF1493; color: white; 
               text-decoration: none; font-size: 16px; border-radius: 8px; font-weight: bold;">{title}</a>'''
    return HTML(html)

display(create_download_link(file_path))

=== 5. INICIANDO MOTOR DE BÚSQUEDA Y SCORING ===
Evaluando 455 bundles con lógica heurística...

✅ ¡SUBMISSION GENERADA! Total filas: 6825
